In [6]:
from dotenv import load_dotenv
import os

load_dotenv()

api_key = os.getenv("GROQ_API_KEY")


In [7]:
from langchain.agents import AgentState

class CustomState(AgentState):
    favourite_colour: str

## Write to state

In [8]:
from langchain.tools import tool, ToolRuntime
from langgraph.types import Command
from langchain.messages import ToolMessage

@tool
def update_favourite_colour(favourite_colour: str, runtime: ToolRuntime) -> Command:
    """Update the favourite colour of the user in the state once they've revealed it."""
    return Command(update={
        "favourite_colour": favourite_colour, 
        "messages": [ToolMessage("Successfully updated favourite colour", tool_call_id=runtime.tool_call_id)]}
        )

In [9]:
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent  # ← back to this
from langgraph.checkpoint.memory import InMemorySaver

model = init_chat_model(
    "llama-3.3-70b-versatile",  # ← official replacement for the decommissioned model
    model_provider="groq",
    temperature=0,  # ← MUST be 0 for reliable tool calling
)

agent = create_agent(
    model=model,
    tools=[update_favourite_colour],
    checkpointer=InMemorySaver(),
    state_schema=CustomState,
)


In [10]:
from langchain.messages import HumanMessage

response = agent.invoke(
    { "messages": [HumanMessage(content="My favourite colour is green")]},
    {"configurable": {"thread_id": "11"}}
)

In [11]:
from pprint import pprint

pprint(response)

{'favourite_colour': 'green',
 'messages': [HumanMessage(content='My favourite colour is green', additional_kwargs={}, response_metadata={}, id='6b8eeefd-158d-4359-9523-98fd43037f16'),
              AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'gj4jn9hcy', 'function': {'arguments': '{"favourite_colour":"green"}', 'name': 'update_favourite_colour'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 19, 'prompt_tokens': 243, 'total_tokens': 262, 'completion_time': 0.04341638, 'completion_tokens_details': None, 'prompt_time': 0.045321153, 'prompt_tokens_details': None, 'queue_time': 0.145773729, 'total_time': 0.088737533}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_ba38bbab80', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019f3da9-c731-7353-899d-85cce5ede4e5-0', tool_calls=[{'name': 'update_favourite_colour', 'args': {'favourite_colour': 'green'}, 

In [12]:
response = agent.invoke(
    { 
        "messages": [HumanMessage(content="Hello, how are you?")],
        "favourite_colour": "green"
    },
    {"configurable": {"thread_id": "10"}}
)

pprint(response)

{'favourite_colour': 'green',
 'messages': [HumanMessage(content='Hello, how are you?', additional_kwargs={}, response_metadata={}, id='eec3e798-dbb8-49ed-a781-2a592cb46544'),
              AIMessage(content="I'm doing well, thanks for asking. Is there something I can help you with or would you like to chat?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 25, 'prompt_tokens': 244, 'total_tokens': 269, 'completion_time': 0.105192257, 'completion_tokens_details': None, 'prompt_time': 0.054934952, 'prompt_tokens_details': None, 'queue_time': 0.037602251, 'total_time': 0.160127209}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_43d97c5965', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019f3da9-e71e-78c0-a296-a79e314c0b88-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 244, 'output_tokens': 25, 'total_tokens': 269})]}


## Read state

In [13]:
@tool
def read_favourite_colour(runtime: ToolRuntime) -> str:
    """Read the favourite colour of the user from the state."""
    try:
        return runtime.state["favourite_colour"]
    except KeyError:
        return "No favourite colour found in state"
    
agent = create_agent(
    model=model,
    tools=[update_favourite_colour, read_favourite_colour],
    checkpointer=InMemorySaver(),
    state_schema=CustomState,
)

In [14]:
response = agent.invoke(
    { "messages": [HumanMessage(content="My favourite colour is green")]},
    {"configurable": {"thread_id": "1"}}
)

pprint(response)

{'favourite_colour': 'green',
 'messages': [HumanMessage(content='My favourite colour is green', additional_kwargs={}, response_metadata={}, id='ba0b0e7a-3347-45e8-9936-ce9c1ebd8249'),
              AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 't37ygjxde', 'function': {'arguments': '{"favourite_colour":"green"}', 'name': 'update_favourite_colour'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 19, 'prompt_tokens': 297, 'total_tokens': 316, 'completion_time': 0.060607359, 'completion_tokens_details': None, 'prompt_time': 0.029294263, 'prompt_tokens_details': None, 'queue_time': 0.036442068, 'total_time': 0.089901622}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_43d97c5965', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019f3daa-1898-7430-bea6-d54e4a3da496-0', tool_calls=[{'name': 'update_favourite_colour', 'args': {'favourite_colour': 'green'},

In [15]:
response = agent.invoke(
    { "messages": [HumanMessage(content="What's my favourite colour?")]},
    {"configurable": {"thread_id": "1"}}
)

pprint(response)

{'favourite_colour': 'green',
 'messages': [HumanMessage(content='My favourite colour is green', additional_kwargs={}, response_metadata={}, id='ba0b0e7a-3347-45e8-9936-ce9c1ebd8249'),
              AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 't37ygjxde', 'function': {'arguments': '{"favourite_colour":"green"}', 'name': 'update_favourite_colour'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 19, 'prompt_tokens': 297, 'total_tokens': 316, 'completion_time': 0.060607359, 'completion_tokens_details': None, 'prompt_time': 0.029294263, 'prompt_tokens_details': None, 'queue_time': 0.036442068, 'total_time': 0.089901622}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_43d97c5965', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019f3daa-1898-7430-bea6-d54e4a3da496-0', tool_calls=[{'name': 'update_favourite_colour', 'args': {'favourite_colour': 'green'},